In [8]:

#A1Task
import pandas as pd
import numpy as np

captains = pd.read_csv(r'C:\Users\AVINASH\Downloads\DataScience Assessment\captains.csv')
doc_events = pd.read_csv(r'C:\Users\AVINASH\Downloads\DataScience Assessment\doc_events.csv')
approvals = pd.read_csv(r'C:\Users\AVINASH\Downloads\DataScience Assessment\approvals.csv')
# Convert timestamps
captains['signup_ts'] = pd.to_datetime(captains['signup_ts'])
doc_events['event_ts'] = pd.to_datetime(doc_events['event_ts'])
approvals['decision_ts'] = pd.to_datetime(approvals['decision_ts'])
cohort = captains[
    (captains['signup_ts'] >= '2026-05-01') &
    (captains['signup_ts'] < '2026-06-01')
].copy()
print("=" * 70)
print("A1 — ONBOARDING FUNNEL")
print("=" * 70)
print(f"\nMay 2026 cohort signups: {len(cohort):,}")
cohort_ids = set(cohort['captain_id'])
cohort_docs = doc_events[
    doc_events['captain_id'].isin(cohort_ids)
].copy()

passed_docs = cohort_docs[
    cohort_docs['event_type'] == 'verification_pass'
].copy()


doc_pass = (
    passed_docs
    .assign(passed=1)
    .pivot_table(
        index='captain_id',
        columns='doc_type',
        values='passed',
        aggfunc='max',
        fill_value=0
    )
    .reset_index()
)


required_docs = [
    'DL',
    'RC',
    'AADHAAR',
    'PERMIT',
    'FITNESS',
    'INSURANCE'
]

for doc in required_docs:
    if doc not in doc_pass.columns:
        doc_pass[doc] = 0



funnel_data = cohort[
    ['captain_id', 'vehicle_type']
].merge(
    doc_pass,
    on='captain_id',
    how='left'
)

for doc in required_docs:
    funnel_data[doc] = (
        funnel_data[doc]
        .fillna(0)
        .astype(int)
    )



funnel_data = funnel_data.merge(
    approvals[
        ['captain_id', 'final_status']
    ],
    on='captain_id',
    how='left'
)

funnel_data['approved'] = (
    funnel_data['final_status'] == 'approved'
).astype(int)


funnel_data['stage_dl'] = (
    funnel_data['DL'] == 1
)
funnel_data['stage_rc'] = (
    funnel_data['stage_dl'] &
    (funnel_data['RC'] == 1)
)

funnel_data['stage_aadhaar'] = (
    funnel_data['stage_rc'] &
    (funnel_data['AADHAAR'] == 1)
)



funnel_data['permit_required'] = (
    funnel_data['vehicle_type'].isin(['Auto', 'Cab'])
)



funnel_data['stage_4'] = np.where(
    funnel_data['permit_required'],
    funnel_data['stage_aadhaar'] &
    (funnel_data['PERMIT'] == 1),

    funnel_data['stage_aadhaar'] &
    (funnel_data['FITNESS'] == 1)
)


funnel_data['stage_5'] = np.where(
    funnel_data['permit_required'],
    funnel_data['stage_4'] &
    (funnel_data['FITNESS'] == 1),

    funnel_data['stage_4']
)



funnel_data['stage_6'] = (
    funnel_data['stage_5'] &
    (funnel_data['INSURANCE'] == 1)
)



total_signups = len(funnel_data)

funnel = pd.DataFrame({
    'Stage': [
        'Signup',
        'DL cleared',
        'RC cleared',
        'Aadhaar cleared',
        '4th required document cleared',
        '5th required document cleared',
        'Insurance cleared',
        'Approved'
    ],

    'Captains': [
        total_signups,
        funnel_data['stage_dl'].sum(),
        funnel_data['stage_rc'].sum(),
        funnel_data['stage_aadhaar'].sum(),
        funnel_data['stage_4'].sum(),
        funnel_data['stage_5'].sum(),
        funnel_data['stage_6'].sum(),
        funnel_data['approved'].sum()
    ]
})


funnel['Lost'] = (
    funnel['Captains'].shift(1) -
    funnel['Captains']
)

funnel.loc[0, 'Lost'] = np.nan



funnel['Overall_Conversion_%'] = (
    funnel['Captains'] /
    total_signups * 100
)


funnel['Stage_Conversion_%'] = (
    funnel['Captains'] /
    funnel['Captains'].shift(1) * 100
)

funnel.loc[0, 'Stage_Conversion_%'] = 100


# Round values
funnel['Overall_Conversion_%'] = (
    funnel['Overall_Conversion_%'].round(2)
)

funnel['Stage_Conversion_%'] = (
    funnel['Stage_Conversion_%'].round(2)
)



print("\nFunnel:")
print("-" * 70)

print(
    funnel.to_string(index=False)
)


approved_count = funnel_data['approved'].sum()

a2o = (
    approved_count /
    total_signups * 100
)

print("\n" + "=" * 70)
print("KEY A1 METRICS")
print("=" * 70)

print(f"Total signups:       {total_signups:,}")
print(f"Approved captains:   {approved_count:,}")
print(f"A2O:                 {a2o:.2f}%")


print("\n" + "=" * 70)
print("VEHICLE TYPE DISTRIBUTION")
print("=" * 70)

print(
    cohort['vehicle_type']
    .value_counts()
    .to_string()
)



print("\n" + "=" * 70)
print("FINAL STATUS DISTRIBUTION")
print("=" * 70)

status_counts = (
    funnel_data['final_status']
    .value_counts(dropna=False)
)

print(status_counts.to_string())


print("\n" + "=" * 70)
print("LAST STAGE REACHED")
print("=" * 70)

last_stage = (
    approvals[
        approvals['captain_id'].isin(cohort_ids)
    ]['last_stage_reached']
    .value_counts(dropna=False)
)

print(last_stage.to_string())



print("\n" + "=" * 70)
print("VEHICLE-SPECIFIC FUNNEL CHECK")
print("=" * 70)

vehicle_funnel = (
    funnel_data
    .groupby('vehicle_type')
    .agg(
        Signups=('captain_id', 'count'),
        DL=('stage_dl', 'sum'),
        RC=('stage_rc', 'sum'),
        Aadhaar=('stage_aadhaar', 'sum'),
        Stage_4=('stage_4', 'sum'),
        Stage_5=('stage_5', 'sum'),
        Insurance=('stage_6', 'sum'),
        Approved=('approved', 'sum')
    )
)

vehicle_funnel['A2O_%'] = (
    vehicle_funnel['Approved'] /
    vehicle_funnel['Signups'] * 100
).round(2)

print(vehicle_funnel)


loss_table = funnel.iloc[1:].copy()

largest_loss = loss_table.loc[
    loss_table['Lost'].idxmax()
]

print("\n" + "=" * 70)
print("BIGGEST FUNNEL LOSS")
print("=" * 70)

print(
    f"{largest_loss['Stage']}: "
    f"{int(largest_loss['Lost']):,} captains lost"
)
funnel.to_csv(
    'A1_funnel_results.csv',
    index=False
)

funnel_data.to_csv(
    'A1_captain_level_data.csv',
    index=False
)

print("\nA1 output files created:")
print(" - A1_funnel_results.csv")
print(" - A1_captain_level_data.csv")

A1 — ONBOARDING FUNNEL

May 2026 cohort signups: 4,580

Funnel:
----------------------------------------------------------------------
                        Stage  Captains   Lost  Overall_Conversion_%  Stage_Conversion_%
                       Signup      4580    NaN                100.00              100.00
                   DL cleared      4049  531.0                 88.41               88.41
                   RC cleared      2941 1108.0                 64.21               72.64
              Aadhaar cleared      2627  314.0                 57.36               89.32
4th required document cleared      1922  705.0                 41.97               73.16
5th required document cleared      1497  425.0                 32.69               77.89
            Insurance cleared       848  649.0                 18.52               56.65
                     Approved       774   74.0                 16.90               91.27

KEY A1 METRICS
Total signups:       4,580
Approved captains:   